# Test of S1-ARD processor

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-766

In [15]:
# Experimental configuration, used only for testing.
# See: https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/rs_dpr_service/utils/settings.py
experimental_config = {
    "local_cluster": {
        "enabled": False, # Use False to disable
        "n_workers": 3,
        "memory_limit": "12GiB",
    },
    "local_files": {
        "local_dir": None, #"/tmp/data", # Use None to disable
        "overwrite_input": False,
        "upload_output": True,
    },
}

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_s1ard(
    # image="ghcr.io/rs-python/dask/s1ard:my-custom-tag", # default tag is :latest
    scale=1,              # number of workers
    big_resources=False,  # provide more ram and cpu
    worker_cores=3,       # number of CPU per worker 
    worker_memory=12,     # memory per worker in GB
)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

Auxip service: http://rs-server-adgs:8000/auxip
PRIP service: http://rs-server-prip:8000/prip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-s1ard.jgaucher.latest': http://dask-s1ard:8000 ...
Create new dask cluster
Dask dashboard for 'dask-s1ard.jgaucher.latest': http://localhost:8704/clusters/0be6ffd4be8c47a3b02d2e735a622ca4/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+-------------+--------+-----------+---------+
| Package     | Client | Scheduler | Workers |
+-------------+--------+-----------+---------+
| cloudpickle | 3.1.1  | 3.1.2     | None    |
| lz4         | 4.4.4  | 4.4.5     | None    |
| msgpack     | 1.1.0  | 1.1.2     | None    |
| toolz       | 1.0.0  | 1.1.0     | None    |
| tornado     | 6.3.3  | 6.5.2     | None    |
+-------------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-s1ard.jgaucher.latest' are up: 0/1
Dask workers for 'dask-s1ard.jgaucher.latest' are up: 1/1


In [9]:
# Other imports
import os.path as osp
from IPython.display import JSON
from resources.dpr_utils import DprDemo
from rs_client.ogcapi.dpr_client import DprProcessor

## Read the tasktables

Documentation: https://cpm.pages.eopf.copernicus.eu/eopf-cpm/main/processor-orchestration-guide/tasktables.html#tasktables

<div class="alert alert-block alert-danger">

Note: not implemented for now for S1-ARD.
</div>

In [10]:
# NOTE: not implemented for now in S1-ARD
# for process in [DprProcessor.S1ARD]:
#     tasktable: dict = dpr_client.get_process(process.value, cluster_info_eopf)
#     print(f"Tasktable for {process.value!r}:")
#     display(JSON(tasktable))
#     # print(json.dumps(tasktable, indent=2))

## Choose the running mode.

In [11]:
import ipywidgets as widgets
from IPython.display import display

payload_map = {
    "Run all units": "s1-ard/demo_joborder.yaml",
    "Calibration IW": "s1-ard/demo_joborder_calibration_iw.yaml",
    "Calibration SM": "s1-ard/demo_joborder_calibration_sm.yaml",
    "DEM IW": "s1-ard/demo_joborder_dem_iw.yaml",
    "DEM SM": "s1-ard/demo_joborder_dem_sm.yaml",
    "Reference Geometry IW": "s1-ard/demo_joborder_ref_geom_iw.yaml",
    "Reference Geometry SM": "s1-ard/demo_joborder_ref_geom_sm.yaml",
    "Co-Registration IW": "s1-ard/demo_joborder_coregistration_iw.yaml",
    "Co-Registration SM": "s1-ard/demo_joborder_coregistration_sm.yaml",
}

dropdown = widgets.Dropdown(
    options=payload_map,
    description="Job type:",
    style={'description_width': 'initial'},
)

display(dropdown)

Dropdown(description='Job type:', options={'Run all units': 's1-ard/demo_joborder.yaml', 'Calibration IW': 's1…

## Init environment for the processors.

In [16]:
# Get the user's choice
payload_subpath = dropdown.value
print(f"Running configuration file: {dropdown.value!r}")
# Init DPR processor demo
dpr = DprDemo(
    owner_id=OWNER_ID, 
    dpr_client=dpr_client,
    local_config_dir="./config"
)
await dpr.init(local_secrets_file="./config/secrets.json")
    
# Same arguments for all tests
dpr_args = {
    "process": DprProcessor.S1ARD,
    "cluster_info": cluster_info_eopf,
    "payload_subpath": payload_subpath,
    "experimental_config": experimental_config,
}

Running configuration file: 's1-ard/demo_joborder_calibration_iw.yaml'


12:34:10.109 | INFO    | prefect.S3Bucket - Uploading from '/home/jovyan/notebooks/sprints/sprint29/test_s1_ard/config/secrets.json' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/config/secrets.json'.

12:34:10.110 | INFO    | prefect.S3Bucket - Uploading from '/home/jovyan/notebooks/sprints/sprint29/test_s1_ard/config/logging_config.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/config/logging_config.yaml'.

12:34:10.112 | INFO    | prefect.S3Bucket - Uploading from '/home/jovyan/notebooks/sprints/sprint29/test_s1_ard/config/s1-ard/demo_joborder_ref_geom_sm.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/config/s1-ard/demo_joborder_ref_geom_sm.yaml'.

12:34:10.113 | INFO    | prefect.S3Bucket - Uploading from '/home/jovyan/notebooks/sprints/sprint29/test_s1_ard/config/s1-ard/demo_joborder_dem_iw.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/config/s1-ard/demo_joborder_dem_iw.yaml'.

...

12:34:10.117 | INFO    | prefect.S3Bucket - Uploading from '/home/jovyan/notebooks/sprints/sprint29/test_s1_ard/config/s1-ard/demo_joborder_coregistration_iw.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/config/s1-ard/demo_joborder_coregistration_iw.yaml'.

12:34:10.118 | INFO    | prefect.S3Bucket - Uploading from '/home/jovyan/notebooks/sprints/sprint29/test_s1_ard/config/s1-ard/demo_joborder_calibration_sm.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/config/s1-ard/demo_joborder_calibration_sm.yaml'.

12:34:10.119 | INFO    | prefect.S3Bucket - Uploading from '/home/jovyan/notebooks/sprints/sprint29/test_s1_ard/config/s1-ard/demo_joborder_coregistration_sm.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/config/s1-ard/demo_joborder_coregistration_sm.yaml'.

12:34:10.158 | INFO    | prefect.S3Bucket - Uploaded 11 files from '/home/jovyan/notebooks/sprints/sprint29/test_s1_ard/config' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/config/s1-ard/demo_joborder_coregistration_sm.yaml'

12:34:10.166 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpwl4oxo33' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/config/secrets.json'.

## Run processor

In [13]:
# Clean the working dir in the s3 bucket ?
clean_working_dir = True

In [17]:
# Run S1-ARD
if os.getenv("RSPY_FROM_CICD") != "1":
    s3_output_dir = osp.join(dpr.s3_output_dir, "s1ard")
    s3_working_dir = osp.join(dpr.s3_working_dir, "s1ard")
    await dpr.run_process(
        **dpr_args,
        s3_output_dir = s3_output_dir,
        s3_report_dir = osp.join(dpr.s3_report_dir, "s1ard"),
        del_s3_working_dir = s3_working_dir if clean_working_dir else "",
        # Payload env vars
        OUTPUT_DIR = s3_output_dir,
        WORKING_DIR = s3_working_dir,
    )

s3_config_dir: s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/config
payload_subpath: s1-ard/demo_joborder_calibration_iw.yaml
Remove s3_output_dir: s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/output/s1ard
Remove s3_report_dir: s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/reports/s1ard


12:34:12.159 | INFO    | prefect.S3Bucket - Delete from 'http://minio:9000': [
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/output/s1ard/S01SIWCSL_20240416T171518_0027_A015_S000_00000_DV.zarr/.empty",
  "s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/output/s1ard/S01SIWNRB_20240416T171518_0027_A015_S000_00000_DV.zarr/.empty"
]

Remove s3_working_dir: s3://rs-dev-cluster-temp/prefect-share/users/jgaucher/working/s1ard


12:34:12.443 [INFO] (rs_client.rs_client) Write empty file: 20 's3://rs-dev-cluster-temp/prefect-share/users/jgaucher/output/s1ard/S01SIWCSL_20240416T171518_0027_A015_S000_00000_DV.zarr/.empty'


12:34:12.460 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmp78jk2suc' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/output/s1ard/S01SIWCSL_20240416T171518_0027_A015_S000_00000_DV.zarr/.empty'.

...

12:34:12.470 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmp2t21r212' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/output/s1ard/S01SIWNRB_20240416T171518_0027_A015_S000_00000_DV.zarr/.empty'.

12:34:12.479 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpo52rbwxz' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/jgaucher/config/s1-ard/demo_joborder_calibration_iw.yaml'.

12:34:12.670 [INFO] (resources.dpr_utils) job_status: {'progress': 0, 'created': '2025-11-25T12:34:12Z', 'started': '2025-11-25T12:34:12Z', 'updated': '2025-11-25T12:34:12Z', 'status': 'running', 'processID': 'dpr-service', 'type': 'process', 'message': 'Sending task to the dask cluster', 'jobID': 'ebddfef0-c476-481a-bab0-fbf0c18ca890'}
12:34:12.671 [INFO] (resources.dpr_utils) -----  job 'ebddfef0-c476-481a-bab0-fbf0c18ca890': RUNNING 

12:34:17.692 [INFO] (resources.dpr_utils) job_status: {'progress': 50, 'created': '2025-11-25T12:34:12Z', 'started': '2025-11-25T12:34:12Z', 'updated': '2025-11-25T12:34:12Z', 'status': 'running', 'processID': 'dpr-service', 'type': 'process', 'message': 'In progress', 'jobID': 'ebddfef0-c476-481a-bab0-fbf0c18ca890'}
12:34:17.693 [INFO] (resources.dpr_utils) -----  job 'ebddfef0-c476-481a-bab0-fbf0c18ca890': RUNNING 

12:34:22.708 [INFO] (resources.dpr_utils) job_status: {'progress': 50, 'created': '2025-11-25T12:34:12Z', 'started': '2025-11-25T12:34:1

Processor execution time: 0:07:51.592089
Log file './reports/s1ard/demo_joborder_calibration_iw.processor.log':
Dask cluster label: 'dask-s1ard.jgaucher.latest'
Dask cluster instance: '0be6ffd4be8c47a3b02d2e735a622ca4'
Payload file contents: '/tmp/eopf-cz4uuu8d/config/s1-ard/demo_joborder_calibration_iw.yaml'
{
  "I/O": {
    "adfs": [
      {
        "id": "CONFIG",
        "path": "s3://rs-dev-cluster-temp/ARD_V2/ADFS/CONFIG/ard.json",
        "store_params": {
          "storage_options": {
            "client_kwargs": {
              "endpoint_url": ***,
              "region_name": ***
            },
            "key": ***,
            "secret": ***
          }
        }
      },
      {
        "id": "S2_TILES",
        "path": "s3://rs-dev-cluster-temp/ARD_V2/ADFS/KML/S2A_OPER_GIP_TILPAR_MPC__20151209T095117_V20150622T000000_21000101T000000_B00.kml",
        "store_params": {
          "storage_options": {
            "client_kwargs": {
              "endpoint_url": ***,
       

RuntimeError:  job 'ebddfef0-c476-481a-bab0-fbf0c18ca890': FAILED

## Shutdown cluster

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.